# Safety in Spanish: A Cross-Lingual Evaluation of Jailbreak Vulnerability in LLMs

## Experiments

**Purpose:** Run all benchmark prompts through target models (HuggingFace + API), save one CSV per model, then combine for evaluation.

> Note: Each model section is **fully independent**

## 0. Setup

### 0.1 Install Dependencies

In [ ]:
!pip install langdetect pydantic openai anthropic transformers accelerate bitsandbytes --quiet

### 0.2 Imports

In [3]:
import csv
import gc
import json
import logging
import os
import sys
import time
from datetime import datetime
from pathlib import Path
from typing import Optional

import pandas as pd
import torch
from IPython.display import display
from langdetect import detect_langs, LangDetectException
from pydantic import BaseModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

import anthropic
import openai
from google.colab import userdata
from tqdm.auto import tqdm

### 0.3 Configuration

In [4]:
BENCHMARK_PATH = Path("/content/benchmark.csv")
RESULTS_DIR = Path("/content")
LOGS_DIR = Path("/content/logs")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

TEMPERATURE = 1.0
MAX_NEW_TOKENS = 1500

HF_N_RUNS = 5
API_N_RUNS = 3

USE_4BIT_QUANTIZATION = False
HF_BATCH_SIZE = 16
HF_MAX_INPUT_LEN = 1024

BUDGET_CAP_USD = 45.0
SKIP_COST_CONFIRMATION = False

### 0.4 Logging

In [5]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOGS_DIR / "experiments.log"),
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger(__name__)
logger.info("Logging initialised")

### 0.5 Secrets

In [6]:
HF_TOKEN = userdata.get("HF_TOKEN")

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

logger.info("Secrets loaded")

## 1. Data Schema

In [7]:
class Response(BaseModel):
    response_id: str
    prompt_id: str
    model: str
    run_index: int
    temperature: float
    response_text: str
    response_language: str
    tokens_input: int
    tokens_output: int
    cost_usd: float
    timestamp: str
    error: Optional[str] = None

## 2. Dataset

Load benchmark once. All runners reference `df_benchmark`.

In [ ]:
assert BENCHMARK_PATH.exists(), f"benchmark.csv not found at {BENCHMARK_PATH}"

df_benchmark = pd.read_csv(BENCHMARK_PATH)

_required = {"prompt_id", "source", "category", "base_text_en", "text", "language", "attack_type"}
_missing = _required - set(df_benchmark.columns)
assert not _missing, f"Missing columns: {_missing}"

logger.info(f"Loaded {len(df_benchmark)} rows from benchmark.csv")
logger.info(f"Language : {df_benchmark['language'].value_counts().to_dict()}")
logger.info(f"Attack   : {df_benchmark['attack_type'].value_counts().to_dict()}")
logger.info(f"Category : {df_benchmark['category'].value_counts().to_dict()}")
display(df_benchmark.head())

## 3. Runner Classes

### 3.1 Shared Utilities

In [10]:
def detect_response_language(text: str) -> str:
    if not text or len(text.strip()) < 15:
        return "other"
    try:
        langs = detect_langs(text)
        prob = {e.lang: e.prob for e in langs}
        if prob.get("en", 0.0) >= 0.7:
            return "en"
        if prob.get("es", 0.0) >= 0.7:
            return "es"
        if prob.get("en", 0.0) >= 0.2 and prob.get("es", 0.0) >= 0.2:
            return "mixed"
        top = langs[0].lang if langs else "other"
        return top if top in {"en", "es"} else "other"
    except LangDetectException:
        return "other"

def make_response_id(prompt_id: str, model: str, run_index: int) -> str:
    slug = model.replace("/", "_").replace(":", "_").replace(".", "_")
    return f"{prompt_id}__{slug}__{run_index}"

def model_csv_path(model_id: str, results_dir: Path) -> Path:
    slug = model_id.replace("/", "_")
    return results_dir / f"responses_{slug}.csv"

def load_completed_keys(output_path: Path) -> set[tuple[str, str, int]]:
    if not output_path.exists():
        return set()
    df = pd.read_csv(output_path)
    return set(zip(df["prompt_id"], df["model"], df["run_index"]))

def append_record(record: dict, output_path: Path) -> None:
    write_header = not output_path.exists()
    with output_path.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(record.keys()))
        if write_header:
            writer.writeheader()
        writer.writerow(record)

### 3.2 HFModelRunner


In [11]:
class HFModelRunner:
    """
    Loads, runs, and unloads a HuggingFace causal LM.
    """
    def __init__(self, model_id: str, n_runs: int, results_dir: Path) -> None:
        self.model_id = model_id
        self.n_runs = n_runs
        self.output_path = model_csv_path(model_id, results_dir)
        self._model: Optional[AutoModelForCausalLM] = None
        self._tokenizer: Optional[AutoTokenizer] = None

    def load_model(self) -> None:
        logger.info(f"Loading {self.model_id} ...")
        tokenizer = AutoTokenizer.from_pretrained(
            self.model_id, token=HF_TOKEN, trust_remote_code=True
        )
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "left"
        quant_config = (
            BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
            )
            if USE_4BIT_QUANTIZATION
            else None
        )
        model_kwargs: dict = dict(
            token=HF_TOKEN,
            trust_remote_code=True,
            device_map="auto",
        )
        if quant_config is not None:
            model_kwargs["quantization_config"] = quant_config
        else:
            model_kwargs["torch_dtype"] = torch.float16
        model = AutoModelForCausalLM.from_pretrained(self.model_id, **model_kwargs)
        model.eval()
        self._model = model
        self._tokenizer = tokenizer
        logger.info(f"Loaded {self.model_id}")

    def unload_model(self) -> None:
        del self._model, self._tokenizer
        self._model = None
        self._tokenizer = None
        torch.cuda.empty_cache()
        gc.collect()
        logger.info(f"Unloaded {self.model_id} — GPU cache cleared")

    def run_predictions(self, df: pd.DataFrame) -> None:
        done_keys = load_completed_keys(self.output_path)
        rows = df.to_dict("records")
        total = len(rows) * self.n_runs
        pending_count = sum(
            1 for row in rows for r in range(self.n_runs)
            if (row["prompt_id"], self.model_id, r) not in done_keys
        )
        logger.info("=" * 60)
        logger.info(f"HFModelRunner: {self.model_id}")
        logger.info(f"  Total: {total}  Skipped: {total - pending_count}  Pending: {pending_count}")
        logger.info(f"  Batch size: {HF_BATCH_SIZE}  |  Output: {self.output_path}")
        logger.info("=" * 60)
        if pending_count == 0:
            logger.info("Nothing to do — all responses already saved")
            return
        start_time = time.time()
        new_count = 0
        for run_idx in range(self.n_runs):
            pending_rows = [
                row for row in rows
                if (row["prompt_id"], self.model_id, run_idx) not in done_keys
            ]
            if not pending_rows:
                continue
            logger.info(f"Run {run_idx}: {len(pending_rows)} prompts pending")
            total_batches = (len(pending_rows) + HF_BATCH_SIZE - 1) // HF_BATCH_SIZE
            pbar = tqdm(range(0, len(pending_rows), HF_BATCH_SIZE), total=total_batches,
                        desc=f"{self.model_id.split("/")[-1]} run {run_idx}", unit="batch")
            for batch_idx, batch_start in enumerate(pbar):
                batch_rows = pending_rows[batch_start : batch_start + HF_BATCH_SIZE]
                formatted = [
                    self._tokenizer.apply_chat_template(
                        [{"role": "user", "content": str(row["text"])}],
                        tokenize=False,
                        add_generation_prompt=True,
                    )
                    for row in batch_rows
                ]
                batch_error: Optional[str] = None
                texts = [""] * len(batch_rows)
                tokens_ins = [0] * len(batch_rows)
                tokens_outs = [0] * len(batch_rows)
                try:
                    inputs = self._tokenizer(
                        formatted,
                        return_tensors="pt",
                        padding=True,
                        truncation=True,
                        max_length=HF_MAX_INPUT_LEN,
                    ).to(self._model.device)
                    n_padded = inputs["input_ids"].shape[1]
                    with torch.no_grad():
                        output_ids = self._model.generate(
                            inputs["input_ids"],
                            attention_mask=inputs["attention_mask"],
                            max_new_tokens=MAX_NEW_TOKENS,
                            temperature=TEMPERATURE,
                            do_sample=True,
                            pad_token_id=self._tokenizer.eos_token_id,
                        )
                    for i in range(len(batch_rows)):
                        tokens_ins[i] = int(inputs["attention_mask"][i].sum().item())
                        generated = output_ids[i][n_padded:]
                        texts[i] = self._tokenizer.decode(generated, skip_special_tokens=True)
                        tokens_outs[i] = int(
                            (generated != self._tokenizer.pad_token_id).sum().item()
                        )
                except Exception as exc:
                    batch_error = repr(exc)
                    logger.exception(f"Batch error run {run_idx} @ {batch_start}: {repr(exc)}")
                for i, row in enumerate(batch_rows):
                    record = Response(
                        response_id=make_response_id(row["prompt_id"], self.model_id, run_idx),
                        prompt_id=row["prompt_id"],
                        model=self.model_id,
                        run_index=run_idx,
                        temperature=TEMPERATURE,
                        response_text=texts[i],
                        response_language=detect_response_language(texts[i]),
                        tokens_input=tokens_ins[i],
                        tokens_output=tokens_outs[i],
                        cost_usd=0.0,
                        timestamp=datetime.utcnow().isoformat(),
                        error=batch_error,
                    ).model_dump()
                    append_record(record, self.output_path)
                    done_keys.add((row["prompt_id"], self.model_id, run_idx))
                    new_count += 1
                elapsed = time.time() - start_time
                rate = new_count / elapsed if elapsed > 0 else 1.0
                pbar.set_postfix({
                    "done": new_count,
                    "total": pending_count,
                    "prompts/s": f"{rate:.2f}",
                })
        logger.info(f"Done: {new_count} new responses saved to {self.output_path}")

### 3.3 OpenAIModelRunner



In [ ]:
class OpenAIModelRunner:
    """
    Runs prompts through an OpenAI chat model.
    """

    COST_PER_INPUT_TOKEN = 0.15 / 1_000_000
    COST_PER_OUTPUT_TOKEN = 0.60 / 1_000_000

    def __init__(self, model_id: str, n_runs: int, results_dir: Path) -> None:
        self.model_id = model_id
        self.n_runs = n_runs
        self.output_path = model_csv_path(model_id, results_dir)
        self._total_cost: float = 0.0

    def _predict(
        self,
        client: openai.OpenAI,
        prompt_text: str,
    ) -> tuple[str, int, int, float]:
        completion = client.chat.completions.create(
            model=self.model_id,
            messages=[{"role": "user", "content": prompt_text}],
            temperature=TEMPERATURE,
            max_tokens=MAX_NEW_TOKENS,
        )
        text = completion.choices[0].message.content or ""
        tokens_in = completion.usage.prompt_tokens
        tokens_out = completion.usage.completion_tokens
        cost = (
            tokens_in * self.COST_PER_INPUT_TOKEN
            + tokens_out * self.COST_PER_OUTPUT_TOKEN
        )
        return text, tokens_in, tokens_out, cost

    def run_predictions(self, df: pd.DataFrame) -> None:
        done_keys = load_completed_keys(self.output_path)
        rows = df.to_dict("records")
        pending = [
            (row, r)
            for row in rows
            for r in range(self.n_runs)
            if (row["prompt_id"], self.model_id, r) not in done_keys
        ]

        n_total = len(rows) * self.n_runs
        avg_in, avg_out = 300, 200
        est_cost = len(pending) * (
            avg_in * self.COST_PER_INPUT_TOKEN + avg_out * self.COST_PER_OUTPUT_TOKEN
        )

        logger.info("=" * 60)
        logger.info(f"OpenAIModelRunner: {self.model_id}")
        logger.info(f"  Total calls : {n_total}  |  Pending: {len(pending)}")
        logger.info(f"  Output      : {self.output_path}")
        logger.info("=" * 60)
        print(f"[COST ESTIMATE] {self.model_id}")
        print(f"  Pending calls  : {len(pending)}")
        print(f"  Estimated cost : ${est_cost:.4f}")
        print(f"  Budget cap     : ${BUDGET_CAP_USD:.2f}")

        if not pending:
            logger.info("Nothing to do — all responses already saved")
            return

        if not SKIP_COST_CONFIRMATION:
            if input("  Proceed? [y/N]: ").strip().lower() != "y":
                logger.info("Aborted by user")
                return

        client = openai.OpenAI()
        start_time = time.time()

        for i, (row, run_idx) in enumerate(pending):
            if self._total_cost >= BUDGET_CAP_USD:
                logger.error(f"Budget cap ${BUDGET_CAP_USD} reached — aborting")
                break

            error_msg: Optional[str] = None
            response_text = ""
            tokens_in, tokens_out, call_cost = 0, 0, 0.0

            try:
                response_text, tokens_in, tokens_out, call_cost = self._predict(
                    client, str(row["text"])
                )
                self._total_cost += call_cost
            except Exception as exc:
                error_msg = str(exc)
                logger.error(f"OpenAI error — {row['prompt_id']} run {run_idx}: {exc}")

            record = Response(
                response_id=make_response_id(row["prompt_id"], self.model_id, run_idx),
                prompt_id=row["prompt_id"],
                model=self.model_id,
                run_index=run_idx,
                temperature=TEMPERATURE,
                response_text=response_text,
                response_language=detect_response_language(response_text),
                tokens_input=tokens_in,
                tokens_output=tokens_out,
                cost_usd=call_cost,
                timestamp=datetime.utcnow().isoformat(),
                error=error_msg,
            ).model_dump()
            append_record(record, self.output_path)
            done_keys.add((row["prompt_id"], self.model_id, run_idx))

            n_done = i + 1
            if n_done % 50 == 0:
                logger.info(
                    f"[COST] {self.model_id} — {n_done} calls | "
                    f"cumulative: ${self._total_cost:.4f} / cap ${BUDGET_CAP_USD:.2f}"
                )
            if n_done % 100 == 0:
                elapsed = time.time() - start_time
                rate = n_done / elapsed if elapsed > 0 else 1.0
                logger.info(
                    f"[PROGRESS] {self.model_id} — {n_done}/{len(pending)} "
                    f"({100 * n_done / len(pending):.1f}%) | ETA ~{(len(pending) - n_done) / rate:.0f}s"
                )

        logger.info(
            f"Done: {self.model_id} | total cost this run: ${self._total_cost:.4f}"
        )

### 3.4 AnthropicModelRunner

In [ ]:
class AnthropicModelRunner:
    """
    Runs prompts through an Anthropic Claude model.
    """

    COST_PER_INPUT_TOKEN = 1.00 / 1_000_000
    COST_PER_OUTPUT_TOKEN = 5.00 / 1_000_000

    def __init__(self, model_id: str, n_runs: int, results_dir: Path) -> None:
        self.model_id = model_id
        self.n_runs = n_runs
        self.output_path = model_csv_path(model_id, results_dir)
        self._total_cost: float = 0.0

    def _predict(
        self,
        client: anthropic.Anthropic,
        prompt_text: str,
    ) -> tuple[str, int, int, float]:
        message = client.messages.create(
            model=self.model_id,
            max_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            messages=[{"role": "user", "content": prompt_text}],
        )
        text = message.content[0].text if message.content else ""
        tokens_in = message.usage.input_tokens
        tokens_out = message.usage.output_tokens
        cost = (
            tokens_in * self.COST_PER_INPUT_TOKEN
            + tokens_out * self.COST_PER_OUTPUT_TOKEN
        )
        return text, tokens_in, tokens_out, cost

    def run_predictions(self, df: pd.DataFrame) -> None:
        done_keys = load_completed_keys(self.output_path)
        rows = df.to_dict("records")
        pending = [
            (row, r)
            for row in rows
            for r in range(self.n_runs)
            if (row["prompt_id"], self.model_id, r) not in done_keys
        ]

        n_total = len(rows) * self.n_runs
        avg_in, avg_out = 300, 200
        est_cost = len(pending) * (
            avg_in * self.COST_PER_INPUT_TOKEN + avg_out * self.COST_PER_OUTPUT_TOKEN
        )

        logger.info("=" * 60)
        logger.info(f"AnthropicModelRunner: {self.model_id}")
        logger.info(f"  Total calls : {n_total}  |  Pending: {len(pending)}")
        logger.info(f"  Output      : {self.output_path}")
        logger.info("=" * 60)
        print(f"[COST ESTIMATE] {self.model_id}")
        print(f"  Pending calls  : {len(pending)}")
        print(f"  Estimated cost : ${est_cost:.4f}")
        print(f"  Budget cap     : ${BUDGET_CAP_USD:.2f}")

        if not pending:
            logger.info("Nothing to do — all responses already saved")
            return

        if not SKIP_COST_CONFIRMATION:
            if input("  Proceed? [y/N]: ").strip().lower() != "y":
                logger.info("Aborted by user")
                return

        client = anthropic.Anthropic()
        start_time = time.time()
        new_count = 0

        pbar = tqdm(pending, desc=f"{self.model_id} responses", unit="response")
        for i, (row, run_idx) in enumerate(pbar):
            if self._total_cost >= BUDGET_CAP_USD:
                logger.error(f"Budget cap ${BUDGET_CAP_USD} reached — aborting")
                break

            error_msg: Optional[str] = None
            response_text = ""
            tokens_in, tokens_out, call_cost = 0, 0, 0.0

            try:
                response_text, tokens_in, tokens_out, call_cost = self._predict(
                    client, str(row["text"])
                )
                self._total_cost += call_cost
            except Exception as exc:
                error_msg = str(exc)
                logger.error(f"Anthropic error — {row['prompt_id']} run {run_idx}: {exc}")

            record = Response(
                response_id=make_response_id(row["prompt_id"], self.model_id, run_idx),
                prompt_id=row["prompt_id"],
                model=self.model_id,
                run_index=run_idx,
                temperature=TEMPERATURE,
                response_text=response_text,
                response_language=detect_response_language(response_text),
                tokens_input=tokens_in,
                tokens_output=tokens_out,
                cost_usd=call_cost,
                timestamp=datetime.utcnow().isoformat(),
                error=error_msg,
            ).model_dump()
            append_record(record, self.output_path)
            done_keys.add((row["prompt_id"], self.model_id, run_idx))
            new_count += 1

            pbar.set_postfix({"cost_usd": f"{self._total_cost:.4f}"})

        logger.info(
            f"Done: {self.model_id} | total cost this run: ${self._total_cost:.4f}"
        )


## 4. HuggingFace Models



In [ ]:
_MODEL_ID = ""
runner = HFModelRunner(_MODEL_ID, n_runs=HF_N_RUNS, results_dir=RESULTS_DIR)
runner.load_model()
runner.run_predictions(df_benchmark)
runner.unload_model()

#### Download a Local File


In [ ]:
from google.colab import files

PATH_TO_DOWNLOAD = ""

if os.path.exists(PATH_TO_DOWNLOAD):
    files.download(PATH_TO_DOWNLOAD)
else:
    print(f"File not found: {file_to_download}")

## 5. API Models


### 5.1 GPT-4o Mini

In [ ]:
_GPT_ID = "gpt-4o-mini"
runner = OpenAIModelRunner(_GPT_ID, n_runs=API_N_RUNS, results_dir=RESULTS_DIR)
runner.run_predictions(df_benchmark)

### 5.2 Claude Haiku 4.5

In [ ]:
_HAIKU_ID = "claude-haiku-4-5-20251001"
runner = AnthropicModelRunner(_HAIKU_ID, n_runs=API_N_RUNS, results_dir=RESULTS_DIR)
runner.run_predictions(df_benchmark)

## 6. Combine Results


In [ ]:
csv_files = sorted(RESULTS_DIR.glob("responses_*.csv"))
csv_files = [f for f in csv_files if f.name != "responses_all.csv"]

if not csv_files:
    raise FileNotFoundError(f"No per-model CSVs found in {RESULTS_DIR}")

df_parts = []
for path in csv_files:
    df_part = pd.read_csv(path)
    logger.info(f"  {path.name}: {len(df_part)} rows")
    df_parts.append(df_part)

df_all = pd.concat(df_parts, ignore_index=True)

# Sanity check: no duplicate (response_id)
n_dupes = df_all["response_id"].duplicated().sum()
if n_dupes:
    logger.warning(f"Found {n_dupes} duplicate response_ids — check for overlapping runs")

_all_path = RESULTS_DIR / "responses_all.csv"
df_all.to_csv(_all_path, index=False)
logger.info(f"Combined {len(df_all)} responses from {len(csv_files)} files -> {_all_path}")
display(df_all.groupby("model").size().rename("n_responses").to_frame())